# A3 移植 (7/8)：性能账本怎么读 —— ON/OFF 归因

> **一句话结论**：4k bs16 稳态 ON 55.3 vs OFF 38.9 ms/步（+16.4 ms，+42%）；账本按项归因，先分清哪项是结构性成本、哪项是测量噪声。

## 1. 对比矩阵读法

- **OFF**：关掉 HiSparse 的基线臂；
- **bf16 桥**：走 bf16 直读；
- **int8 直读**：M1 桥接后的 int8 路径；
- 新机 ON/OFF：同日同机 A/B，避免混型号混批次比较。

## 2. 稳态归因账本（4k bs16）

| 项 | ms/步 | 说明 |
| --- | --- | --- |
| Resolver AICPU | +5.24 | 20.8 × 264 µs，下一最大单项 |
| int8 KV 行构建（M4 后） | ≈+3.4 | 融合后残留；prefill 未融合 |
| 注意力净差 | +2.44 | int8 SFA 122 µs/层 vs bf16 91 µs/层（1.34×） |
| 量化 Q 路径 | +2.10 | QuantBatchMatmulV3 254 → 334 次/步 |
| reduce_scatter | +1.31 | 3.22 vs 1.91 |
| gather_rows | +0.66 | 77.8 × 9 µs |
| invalidate | +0.55 | 0.8 × 656 µs |
| 其它 | +0.7 | 余量项 |

## 3. 口径注意

- 主口径是 **bs16**；bs1–15 受图 padding 税影响；
- 16k/32k × bs16 的 churn 格两臂轮间散布 **±10 ms**，属噪声；融合收益需 fresh-server + 受控 churn 协议才能定论；
- 软件观测口径不等同物理 DMA 实测；
- pack scale 用 `Muls` 倒数，与 vendor `amax/127` 有 1 ULP 差；codes ≤7/8192 差 ±1，属量化噪声级。

## 4. 小结

- 最大结构性成本：resolver AICPU、int8 行构建、int8 SFA 慢 1.34×、量化 Q；
- 读账本先排除 padding 税与 churn 噪声；
- 最后一篇看真实踩坑清单与遗留方向。

**下一篇**：`08-pitfalls-and-next.ipynb`
